<a href="https://colab.research.google.com/github/FinestMaximus/mrkt_screener/blob/main/Screening_Stocks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Getting Requirements**

In [19]:
! pip install matplotlib pandas mplfinance marketprofile plotly
! pip install yfinance --upgrade --no-cache-dir

# **Imports**

In [15]:
# Standard library imports
from datetime import datetime, timedelta

# Third-party imports
import matplotlib.pyplot as plt
import mplfinance as mpf
import numpy as np
import time
from tqdm import tqdm
import random
import pandas as pd
import pandas_datareader as pdr
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Markdown, display
import yfinance as yf

# Local application/library specific imports
from market_profile import MarketProfile

# **Get all Companies (~15000)**

In [13]:
file_path = '/content/drive/My Drive/perso.csv'

try:
    data = pd.read_csv(file_path, encoding='ISO-8859-1')

    expected_columns = {'ticker', 'exchange'}
    if set(data.columns) & expected_columns == expected_columns:
        print('CSV file loaded successfully with the expected headers.')
        companies = data['ticker'].tolist()
    else:
        missing_columns = expected_columns - set(data.columns)
        print(f'CSV file is missing the following required columns: {missing_columns}')
        companies = []
except Exception as e:
    print(f'An error occurred while loading the CSV file with an alternative encoding: {e}')
    companies = []

print(len(companies))

cache_folder = '/content/drive/My Drive/colab_cache/'

CSV file loaded successfully with the expected headers.
15417


# **Parametters**

In [2]:
# Time period settings
days_history = 365 * 1  # looking 5 years back
interval_dates = '3mo'  # each candle represents a quarter

# Company performance indicators
eps_threshold = 5  # interested in high revenue companies
gross_margin_threshold = 50  # minimum acceptable gross margin

# Valuation metrics
peg_threshold_low = -0.1  # anything under 0 is overvalued or is not cash generating
peg_threshold_high = 1.1  # anything over 1 is overvalued

# **Get Date Ranges**

In [6]:
def get_date_range(days_back):
    """Helper function to compute start and end date strings."""
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days_back)
    return start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d')

start_date_str, end_date_str = get_date_range(days_history)

# **Data fetching and preparation**

In [20]:
def populate_metrics(ticker, metrics):
    """
    Populates a given metrics dictionary with data fetched from a ticker object.
    """
    if ticker and hasattr(ticker, 'info'):
        stock_info = ticker.info

        metrics['eps_values'].append(stock_info.get('trailingEps', 0))
        metrics['pe_values'].append(stock_info.get('trailingPE', 0))
        metrics['peg_values'].append(stock_info.get('pegRatio', 0))
        metrics['gross_margins'].append(stock_info.get('grossMargins', 0))
        metrics['company_labels'].append(ticker.ticker)

        metrics['companies_fetched'] += 1
    else:
        print(f"Skipped a company ticker due to missing info or an invalid object.")

def populate_metrics(ticker, metrics):
    """
    Placeholder function to simulate populating metrics for a ticker.
    Actual implementation would vary based on metric requirements.
    """
    # Sample logic (pseudocode)
    # metrics['eps_values'].append(ticker.info['eps'])
    metrics['companies_fetched'] += 1  # Simulating metrics population

def fetch_metrics_data(companies):
    """
    Fetches and structures various financial metrics for the given list of company tickers.
    """
    companies_str = [str(company) for company in companies]
    tickers = yf.Tickers(' '.join(companies_str))

    metrics = {metric: [] for metric in ['eps_values', 'pe_values', 'peg_values', 'gross_margins', 'company_labels']}
    metrics['market_caps'] = []
    metrics['ps_values'] = []
    metrics['pb_values'] = []
    metrics['recommendations_summary'] = []
    metrics['price_diff'] = {}
    metrics['companies_fetched'] = 0

    total_companies = len(companies)
    chunk_size = 1000
    break_intervals = [60, 120, 300, 600]
    total_chunks = (total_companies - 1) // chunk_size + 1

    for i in range(0, total_companies, chunk_size):
        chunk_number = (i // chunk_size) + 1
        print(f'Starting processing of Chunk {chunk_number}/{total_chunks}...')
        chunk = companies[i:i + chunk_size]

        for company in chunk:
            try:
                ticker = tickers.tickers[str(company)]
                # Simulating a delay to mimic real-world API call pacing
                time.sleep(1)
                populate_metrics(ticker, metrics)
            except KeyError:
                print(f"Ticker {company} not found. Skipping...")
                continue

        if i + chunk_size < total_companies:
            break_time = random.choice(break_intervals)
            print(f"Taking a break for {break_time//60} minute(s) to avoid throttling...")
            time.sleep(break_time)

        print(f'Finished processing of Chunk {chunk_number}/{total_chunks}.')

    print(f"\nTotal companies fetched: {metrics['companies_fetched']}/{total_companies}")
    return metrics

def update_csv_with_metrics(file_path, metrics):
    """
    Updates the CSV file with the fetched metrics data.

    Args:
    - file_path: Path to the CSV file to be updated.
    - metrics: Dictionary containing arrays of financial metrics.
    """
    # Convert the metrics dictionary into a DataFrame, fixing columns for clarity
    metrics_df = pd.DataFrame({
        'eps': metrics['eps_values'],
        'pe': metrics['pe_values'],
        'peg': metrics['peg_values'],
        'gross_margin': metrics['gross_margins']    # Adjust the name as per your requirement
    })

    # Load the original CSV
    original_df = pd.read_csv(file_path, encoding='ISO-8859-1')

    # If we are appending new information to the original DataFrame based on 'ticker',
    # a left join ensures that all original data is preserved and new columns are added.
    updated_df = pd.merge(original_df, metrics_df, on='ticker', how='left')

    # Now, let's save the updated DataFrame back to the CSV file
    updated_df.to_csv(file_path, encoding='ISO-8859-1', index=False)
    print('CSV file has been updated with the fetched metrics.')

metrics = fetch_metrics_data(companies)
update_csv_with_metrics(file_path, metrics)

AttributeError: 'YFinance' object has no attribute 'Tickers'

# **Filtering Methods**

In [7]:
def filter_companies(metrics, eps_threshold, peg_threshold_low, peg_threshold_high, gross_margin_threshold):
    try:
        # Create DataFrame from metrics
        df = pd.DataFrame({
            'company': metrics['company_labels'],
            'eps': metrics['eps_values'],
            'pe': metrics['pe_values'],
            'ps': metrics['ps_values'],
            'pb': metrics['pb_values'],
            'peg': metrics['peg_values'],
            'gross_margin': metrics['gross_margins']
        })

        # Normalize gross margin if max is less than or equal to 1 (assuming it's in decimals rather than percentage)
        if df['gross_margin'].max() <= 1:
            df['gross_margin'] *= 100

        print(df)

        # Criteria for filtering - each condition needs to be enclosed in parentheses
        criteria = (df['eps'] > eps_threshold) & \
                   (df['gross_margin'] > gross_margin_threshold) & \
                   (df['peg'] > peg_threshold_low) & (df['peg'] <= peg_threshold_high)

        # Apply criteria to filter DataFrame
        filtered_df = df[criteria]

        # Sort filtered DataFrame by 'pe' in ascending order
        filtered_df_sorted = filtered_df.sort_values(by='pe', ascending=True)

        # Print the filtered DataFrame
        print(f"Filtered down to {len(filtered_df_sorted)} companies based on criteria.")
        print(filtered_df_sorted)

        return filtered_df_sorted
    except Exception as e:
        print(f"An error occurred: {e}")
        return pd.DataFrame()

def classify_by_industry(tickers):
    industries = {}
    for ticker in tickers.tickers.values():
        sector = ticker.info.get('sector')
        if sector:
            industries.setdefault(sector, []).append(ticker.ticker)
    return industries

def fetch_industries(companies):
    tickers = yf.Tickers(' '.join(companies))
    industries = classify_by_industry(tickers)
    return industries

def fetch_recommendations_summary(ticker):
    """Fetch a structured summary of recommendations for a given ticker."""
    try:
        rec_data = ticker.get_recommendations_summary()
        if not rec_data.empty:
            return {row['period']: {
                'strongBuy': row['strongBuy'],
                'buy': row['buy'],
                'hold': row['hold'],
                'sell': row['sell'],
                'strongSell': row['strongSell']
            } for index, row in rec_data.iterrows()}
        else:
            return "No recommendation data available."
    except Exception as e:
        return f"Error: {str(e)}"

def fetch_company_news(ticker):
    """Fetches the latest news related to a given ticker."""
    try:
        news_articles = ticker.news
        organized_news = []
        for article in news_articles:
            organized_article = {
                'title': article['title'],
                'publisher': article['publisher'],
                'link': article['link'],
                'published_date': article['providerPublishTime']
            }
            organized_news.append(organized_article)
        return organized_news
    except Exception as e:
        return f"Error fetching news: {str(e)}"

def populate_additional_metrics(ticker, metrics):
    """Populates a given metrics dictionary with data fetched from a ticker object."""
    stock_info = ticker.info
    metrics['recommendations_summary'].append(fetch_recommendations_summary(ticker))
    metrics['news'] = fetch_company_news(ticker)
    metrics['ps_values'].append(stock_info.get('priceToSalesTrailing12Months', 0))
    metrics['pb_values'].append(stock_info.get('priceToBook', 0))
    metrics['market_caps'].append(stock_info.get('marketCap', 0))


def fetch_additional_metrics_data(companies):
    """Fetches and structures various financial metrics for the given list of company tickers."""
    tickers = yf.Tickers(' '.join(companies))
    metrics = {metric: [] for metric in ['ps_values', 'pb_values',
                                     'market_caps', 'recommendations_summary', 'news']}
    metrics['price_diff'] = {}

    for company in companies:
        try:
            ticker = tickers.tickers[company]
            populate_additional_metrics(ticker, metrics)
        except KeyError:
            print(f"Warning: Ticker {company} not found. Skipping.")

    return metrics

filtered_companies_df = filter_companies(metrics, eps_threshold, peg_threshold_low, peg_threshold_high, gross_margin_threshold)

filtered_company_symbols = filtered_companies_df['company'].tolist()

metrics_filtered = fetch_additional_metrics_data(filtered_company_symbols)

# metrics_filtered = fetch_metrics_data(filtered_company_symbols)

NameError: name 'metrics' is not defined

# **Filters Here**

In [ ]:
filtered_industries = fetch_industries(filtered_company_symbols)
industries = fetch_industries(companies)

# **Scatter Plots**

In [ ]:
def fetch_historical_data(ticker, start_date, end_date, period=None, interval=interval_dates):
    try:
        if period:
            data = ticker.history(period=period, interval=interval)
        else:
            data = ticker.history(start=start_date, end=end_date, interval=interval)
        return data
    except Exception as e:
        print(f"Error fetching data for {ticker}: {e}")
        return pd.DataFrame()  # Return an empty DataFrame

def calculate_price_diff(companies):
    tickers = yf.Tickers(' '.join(companies))
    price_diff = {}  # Store price difference info here

    for company in companies:
        ticker = tickers.tickers[company]
        hist = fetch_historical_data(ticker, None, None, period="1y")
        if not hist.empty:
            today_price = hist['Close'].iloc[-1]
            high_52week = max(hist['High'])
            low_52week = min(hist['Low'])
            high_percent_diff = ((today_price - high_52week) / high_52week) * 100
            low_percent_diff = ((today_price - low_52week) / low_52week) * 100
            price_diff[company] = {'high_diff': -1 * high_percent_diff, 'low_diff': low_percent_diff}

    return price_diff

def fetch_price_diff(companies, metrics_filtered):
    price_diff = calculate_price_diff(companies)
    metrics_filtered['price_diff'] = price_diff

    return metrics_filtered

metrics_filtered = fetch_price_diff(filtered_company_symbols, metrics_filtered)

def plot_combined_interactive(metrics_filtered):
    # Extract data for all plots
    company_labels = metrics_filtered['company_labels']
    eps_values = metrics_filtered['eps_values']
    high_diffs = [metrics_filtered['price_diff'][company]['high_diff'] for company in company_labels]
    low_diffs = [metrics_filtered['price_diff'][company]['low_diff'] for company in company_labels]
    market_caps = metrics_filtered['market_caps']
    pb_values = metrics_filtered['pb_values']
    pe_values = metrics_filtered['pe_values']
    peg_values = metrics_filtered['peg_values']
    ps_values = metrics_filtered['ps_values']
    gross_margins = metrics_filtered['gross_margins']
    recommendations_summary = metrics_filtered['recommendations_summary']

    # Normalize PEG sizes for Plot 4 visualization
    peg_min = min(peg_values)
    peg_max = max(peg_values)
    norm_peg_sizes = [(peg - peg_min) / (peg_max - peg_min) * 30 + 10 for peg in peg_values]

    # Adjust the subplot layout
    fig = make_subplots(rows=3, cols=3,  # Update to 3 rows
                        subplot_titles=("Price Difference % Over the Last Year",
                                        "EPS vs P/E Ratio",
                                        "Gross Margin (%)",
                                        "EPS vs P/B Ratio",
                                        "EPS vs PEG Ratio",
                                        "EPS vs P/S Ratio",
                                        "Upgrades & Downgrades Timeline"),  # Adjusted to be in the third row now
                        specs=[
                            [{}, {}, {}],  # First row: 3 individual cells for plots
                            [{}, {}, {}],  # New second row: 3 individual cells for new plots
                            [{"colspan": 3, "type": "scatter"}, None, None]  # Third row: A plot spanning all 3 columns
                        ],
                        vertical_spacing=0.1  # Adjust spacing if needed for aesthetics
                        )

    colors = {company: f'hsl({(i / len(company_labels) * 360)},100%,50%)' for i, company in enumerate(company_labels)}

    ## sorting the gross margin bar charts
    sorted_data = sorted(zip(company_labels, gross_margins, colors), key=lambda x: x[1], reverse=True)  # Sort by gross_margin in descending order
    sorted_labels, sorted_gross_margins, sorted_colors = zip(*sorted_data)

    for i, company in enumerate(company_labels):
        legendgroup = f"group_{company}"
        marker_size = max(market_caps[i] / max(market_caps) * 50, 5)

        # Plot 1: Price Difference
        fig.add_trace(
            go.Scatter(
                x=[high_diffs[i]],
                y=[low_diffs[i]],
                marker=dict(size=10, color=colors[company]),
                legendgroup=legendgroup,
                name=company,
                hoverinfo='none',  # Disables default hover info, to use hovertemplate completely.
                hovertemplate=f'Company: {company}<br>High Diff: %{{x}}<br>Low Diff: %{{y}}<extra></extra>',  # Custom hover
            ),
            row=1, col=1
        )

        # Plot 2: EPS vs P/E Ratio
        fig.add_trace(
            go.Scatter(
                x=[eps_values[i]],
                y=[pe_values[i]],
                marker=dict(size=marker_size, color=colors[company]),
                legendgroup=legendgroup,
                showlegend=False,
                hoverinfo='none',
                hovertemplate=f'Company: {company}<br>EPS: ${eps_values[i]}<br>P/E Ratio: {pe_values[i]}<extra></extra>',  # Custom hover
            ),
            row=1, col=2
        )

        # Plot 3: Gross Margin Bar Chart
        fig.add_trace(go.Bar(
            x=[company_labels[i]],
            y=[gross_margins[i] * 100],
            marker=dict(color=colors[company]),
            legendgroup=legendgroup,
            showlegend=False,
            width=0.8  # Adjust this value as needed
        ), row=1, col=3)

        # Plot 4:  EPS vs P/B Ratio
        fig.add_trace(
            go.Scatter(
                x=[eps_values[i]],
                y=[pb_values[i]],
                marker=dict(size=marker_size, color=colors[company]),
                legendgroup=legendgroup,
                showlegend=False,
                hoverinfo='none',
                hovertemplate=f'Company: {company}<br>EPS: ${eps_values[i]}<br>P/B Ratio: {pb_values[i]}<extra></extra>',  # Custom hover
            ),
            row=2, col=1
        )

        # Plot 5:  EPS vs PEG Ratio
        fig.add_trace(
            go.Scatter(
                x=[eps_values[i]],
                y=[peg_values[i]],
                marker=dict(size=marker_size, color=colors[company]),
                legendgroup=legendgroup,
                showlegend=False,
                hoverinfo='none',
                hovertemplate=f'Company: {company}<br>EPS: ${eps_values[i]}<br>PEG Ratio: {peg_values[i]}<extra></extra>',
            ),
            row=2, col=2
        )

        # Plot 6:  EPS vs P/S Ratio
        fig.add_trace(
            go.Scatter(
                x=[eps_values[i]],
                y=[ps_values[i]],
                marker=dict(size=marker_size, color=colors[company]),
                legendgroup=legendgroup,
                showlegend=False,
                hoverinfo='none',
                hovertemplate=f'Company: {company}<br>EPS: ${eps_values[i]}<br>P/S Ratio: {ps_values[i]}<extra></extra>',  # Custom hover
            ),
            row=2, col=3
        )

        # Plot 7: Recommendations Summary
        current_recommendations = recommendations_summary[i]

        if isinstance(current_recommendations, dict) and '0m' in current_recommendations:
            ratings = current_recommendations['0m']
            rating_categories = ['strongBuy', 'buy', 'hold', 'sell', 'strongSell']
            rating_values = [ratings.get(category, 0) for category in rating_categories]

            # Preparing data for bar plot
            y_categories = list(rating_categories) # Convert category names to numeric values
            category_names = {index: name for index, name in enumerate(rating_categories)} # Map for axis ticks

            # Use the count as bar height
            bar_heights = rating_values

            # Add bar chart to the subplot
            fig.add_trace(
                go.Bar(
                    x=rating_categories,  # Categories on the x-axis
                    y=bar_heights,  # Corresponding values on the y-axis
                    marker=dict(color=colors[company]),
                    name=company,
                    legendgroup=legendgroup,
                    showlegend=False,
                    text=company,  # Display company names
                    hoverinfo='y+text'  # Show hover text and value
                ),
                row=3, col=1
            )

            fig.update_yaxes(range=[0, max(peg_values)], row=2, col=2)

            # If you want to apply a starting y-axis of 0 universally to all subplots, you could do:
            for row in range(1, 4):  # Adjust the range accordingly based on your actual number of rows
                for col in range(1, 4):  # Adjust the range accordingly based on your actual number of columns
                    # Use conditions or specific logic if some charts shouldn't start from 0
                    fig.update_yaxes(range=[0, "auto"], row=row, col=col)

        else:
            # Handle unexpected data structure
            continue

    # Update axes titles
    titles = [("High Diff (%)", "Low Diff (%)"), ("EPS", "P/E Ratio"), ("Company", "Gross Margin (%)")]
    for col, (x_title, y_title) in enumerate(titles, start=1):
        fig.update_xaxes(title_text=x_title, row=1, col=col)
        fig.update_yaxes(title_text=y_title, row=1, col=col)

    fig.update_xaxes(title_text="Recommendation Type", row=1, col=4)
    fig.update_yaxes(title_text="Number of Recommendations", row=1, col=4)

    fig.update_layout(height=2000)

    # Layout adjustments for readability and aesthetics
    fig.update_layout(
        updatemenus=[
            dict(
                type="buttons",
                direction="left",
                buttons=list([
                    dict(
                        args=[{"visible": "legendonly"}],  # This sets non-selected traces to be hidden.
                        label="Hide All",
                        method="restyle"
                    ),
                    dict(
                        args=[{"visible": True}],  # This makes all traces visible.
                        label="Show All",
                        method="restyle"
                    ),
                ]),
                pad={"r": 10, "t": 10},
                showactive=True,
                x=0,
                xanchor="left",
                y=-0.15,
                yanchor="top"
            ),
        ]
    )
    # Show the combined plot
    fig.show()

# **Plot P/E & EPS**

In [ ]:
plot_combined_interactive(metrics_filtered)

# **Sectorial Analysis**

In [ ]:
def plot_sector_distribution_interactive(industries):
    # Calculate total tickers per sector
    sector_counts = {sector: len(tickers) for sector, tickers in industries.items()}

    # Data for plotting
    labels = list(sector_counts.keys())
    sizes = list(sector_counts.values())

    # Create the pie chart
    fig = go.Figure(data=[go.Pie(labels=labels, values=sizes, hole=.3)])

    # Customizing the plot
    fig.update_layout(
        title_text="Interactive Ticker Distribution by Sector",
        # Add annotations in the center of the donut pies.
        annotations=[dict(text='Sectors', x=0.50, y=0.5, font_size=20, showarrow=False)]
    )

    # Show plot
    return fig

from IPython.core.display import display, HTML

fig1_html = plot_sector_distribution_interactive(industries_filtered).to_html(full_html=False, include_plotlyjs='cdn')
fig2_html = plot_sector_distribution_interactive(industries).to_html(full_html=False, include_plotlyjs='cdn')

combined_html = f"<div style='display: flex; justify-content: space-around;'>{fig1_html}{fig2_html}</div>"


# **Plot Sectors**

In [ ]:
display(HTML(combined_html))

# **Plotting charts**


In [ ]:
def get_eps_pe_pb_ps_peg(ticker):
    try:
        if ticker.ticker in metrics['company_labels']:
            index = metrics['company_labels'].index(ticker.ticker)
            eps = metrics['eps_values'][index]
            pe = metrics['pe_values'][index]
            ps = metrics['ps_values'][index]
            pb = metrics['pb_values'][index]
            peg = metrics['peg_values'][index]

            return eps, pe, ps, pb, peg
        else:
            # This line is crucial as it will inform us if the ticker was not found in our labels list.
            print(f"Ticker '{ticker.ticker}' not found in the labels list.")
            return None, None
    except Exception as e:
        # Catching any other potential error to understand what might have gone wrong.
        print(f"An error occurred: {e}")
        return None, None

def calculate_market_profile(data):
    mp = MarketProfile(data)
    mp_slice = mp[data.index.min():data.index.max()]

    # Access the attributes you're interested in
    va_high, va_low = mp_slice.value_area
    poc_price = mp_slice.poc_price  # Point of control price
    profile_range = mp_slice.profile_range

    return va_high, va_low, poc_price, profile_range

def plot_with_volume_profile(ticker, start_date, end_date):
    # Fetching the historical data
    data = fetch_historical_data(ticker, start_date, end_date)
    eps, pe, ps, pb, peg = get_eps_pe_pb_ps_peg(ticker)

    if not data.empty:
        # Calculate the market profile
        va_high, va_low, poc_price, _ = calculate_market_profile(data)

        # Creating lines as Pandas Series to ensure compatibility
        poc_line = pd.Series(poc_price, index=data.index)
        va_high_line = pd.Series(va_high, index=data.index)
        va_low_line = pd.Series(va_low, index=data.index)

        # Annotations for Value Area and POC
        apds = [mpf.make_addplot(poc_line, type='line', color='red', linestyle='dashed', width=3),
                mpf.make_addplot(va_high_line, type='line', color='blue', linestyle='dashed', width=0.7),
                mpf.make_addplot(va_low_line, type='line', color='blue', linestyle='dashed', width=0.7)]

        title = f"{ticker.info['shortName']}\n\n\n EPS={eps}, P/E={pe}, P/S={ps}, P/B={pb}, PEG ratio={peg}\n Price of Control in Red, and Value Area in Blue\n Low P/x means company is undervalued \na High EPS means company is profitable\n PEG is typically lower than 1.0\n\n\n"

        # Plotting the chart with MPLFinance
        mpf.plot(data, type="candle", addplot=apds, title=title, style="yahoo", volume=True, show_nontrading=False)
    else:
        print(f"No data found for {ticker.ticker} in the given date range.")

from IPython.display import display, Markdown
import matplotlib.pyplot as plt


# **Plot Filtered Charts**

In [ ]:
plot_sector_charts(industries_filtered, start_date_str, end_date_str)
print(industries_filtered)